In [1]:
import openai

# Print openai version
print(openai.__version__)

1.109.1


In [2]:
# Load the API key from the .env file
from dotenv import load_dotenv
import os

# Load the .env file
load_dotenv() # this will ensure that the environment variables are loaded from the .env file into the system's environment variables.

# Get the API key from the environment variables
openai.api_key = os.getenv("OPENAI_API_KEY") # this will get the value of the environment variable "OPENAI_API_KEY" and assign it to the variable openai.api_key. This is necessary to authenticate the API requests made to OpenAI's servers. Without this key, you won't be able to access the OpenAI models.

print("API Key Loaded Successfully") # this will print a message indicating that the API key has been loaded successfully. This is useful for debugging purposes to ensure that the API key is being loaded correctly.
# print("API Key:", openai.api_key) # this will print the value of the API key. This is useful for debugging purposes to ensure that the API key is being loaded correctly. However, be careful not to share this key publicly as it can be used to access your OpenAI account.

API Key Loaded Successfully


In [3]:
from openai import OpenAI
client = OpenAI(api_key=openai.api_key)

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a 4 line poem about the beauty of nature."}
    ]
)

# print(completion)
print(completion.choices[0].message.content)

In whispering winds, the tall trees sway,  
The golden sun spills light on the bay.  
Petals unfurl to greet the day,  
Nature's beauty forever in display.


In [4]:
from openai import OpenAI
client = OpenAI(api_key=openai.api_key)

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is todays date in New York City?"}
    ]
)

# print(completion)
print(completion.choices[0].message.content)

Today's date in New York City is October 24, 2023.


In [5]:
from openai import OpenAI
client = OpenAI(api_key=openai.api_key)

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is todays date in New York City?"}
    ]
)

# print(completion)
print(completion.choices[0].message.content)

Today's date in New York City is November 2, 2023.


In [6]:
from openai import OpenAI
client = OpenAI(api_key=openai.api_key)

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is todays date in New York City?"}
    ]
)

# print(completion)
print(completion.choices[0].message.content)

Today's date in New York City is October 27, 2023.




---

## **Problem Statement**

In many real-world scenarios, users seek contextual information about the current date and time as part of their natural language queries. For example:

* *“What is today’s date?”*
* *“Can you tell me the current time before I schedule a meeting?”*

While large language models (LLMs) can generate approximate or static answers, they are **not inherently aware of real-time system information**. As a result, they may provide outdated or fabricated responses when asked about the current date or time.

To ensure accuracy, an intelligent assistant must:

1. **Understand natural language queries referring to temporal context.**
2. **Automatically invoke tools that fetch real-time system date and time.**
3. **Integrate tool outputs into a conversational, human-friendly response.**

This functionality is crucial in **productivity tools, scheduling assistants, personal reminders, and business applications**, where real-time awareness significantly enhances user experience and reliability.

---

## **Objective**

Design and implement a **LangChain-powered agent** that integrates real-time utilities for date and time. The agent should:

* **Detect** when a user query involves current date or time.
* **Automatically call the appropriate tool** (`get_current_date` or `get_current_time`).
* **Leverage LangChain’s tool-calling framework** to handle reasoning and orchestration.
* **Maintain a reasoning trace** using `agent_scratchpad` for clarity.
* **Provide responses** in a conversational and context-aware format, ensuring outputs are both precise and user-friendly.

---



In [7]:
# Langchain Simple Agent

# Import required libraries

# For calling the Openai API
from langchain_openai import ChatOpenAI # LLM BRAIN - ENGINE

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # MESSAGES - CHASSIS
# ChatPromptTemplate: It is used to create a prompt template for chat models. This helps in structuring the input for the model.
# MessagesPlaceholder: It is used to indicate where the messages should be placed in the prompt template. This allows for dynamic insertion of messages during the conversation.

from langchain.tools import tool # HANDS - DASHBOARD
# tool: It is used to define a tool that can be called by the agent. This allows the agent to perform specific actions or retrieve information.

from langchain.agents import create_tool_calling_agent, AgentExecutor
# create_tool_calling_agent: It is used to create an agent that can call tools. This allows the agent to use the defined tools to perform tasks.
# AgentExecutor: It is used to execute the agent. This allows the agent to process inputs

In [8]:
# Load the environment variables from the .env file
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [9]:
llm = ChatOpenAI(model="gpt-4o")
# https://python.langchain.com/docs/integrations/chat/openai/

In [10]:
# Step 5: Create prompt template (IMPORTANT: include input & agent_scratchpad)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant. You can use tools whenever necessary to answer user questions."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # Required
    # MessagesPlaceholder - A special placeholder used by LangChain agents to store: a. Intermediate reasoning steps, b. Tool calls, c. Tool outputs - ReAct Style (Thoughts, Actions, and Observations)
])

In [18]:
from datetime import datetime, timezone

now_utc = datetime.now(timezone.utc)

print("Current date (UTC):", now_utc.date())
print("Current time (UTC):", now_utc.strftime("%H:%M:%S %Z"))

Current date (UTC): 2026-11-03
Current time (UTC): 18:56:23 UTC


In [19]:
# Tool Definitions
from datetime import datetime, timezone

@tool
def get_current_utc_date():
    """Returns the current date in UTC."""
    return f"Current date in UTC: {datetime.now(timezone.utc).date()}"

@tool
def get_current_utc_time():
    """Returns the current time in UTC."""
    return f"Current time in UTC: {datetime.now(timezone.utc).strftime('%H:%M:%S %Z')}"

tools = [get_current_utc_date, get_current_utc_time]

In [20]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [21]:
# Wrap the agent in an AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # Set verbose=True to see the agent's reasoning and tool calls in the output
)

In [14]:
response = agent_executor.invoke({"input": "What is the capital of India?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of India is New Delhi.

> Finished chain.
The capital of India is New Delhi.


In [ ]:
response = agent_executor.invoke({"input": "What is today's date in New York city?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date` with `{}`


Current date in UTC: 2026-11-03Today's date in New York City is November 3, 2026.

> Finished chain.
Today's date in New York City is November 3, 2026.


In [24]:
response = agent_executor.invoke({"input": "What is current time in New York city?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date` with `{}`


Current date in UTC: 2026-11-03
Invoking: `get_current_utc_time` with `{}`


Current time in UTC: 19:00:58 UTCThe current UTC time is 19:00:58 on November 3, 2026. New York is typically 5 hours behind UTC. Therefore, the current time in New York City is 14:00:58 (2:00 PM) on November 3, 2026. However, please note that New York may be observing daylight saving time, which would make it 4 hours behind UTC, resulting in a time of 15:00:58 (3:00 PM).

> Finished chain.
The current UTC time is 19:00:58 on November 3, 2026. New York is typically 5 hours behind UTC. Therefore, the current time in New York City is 14:00:58 (2:00 PM) on November 3, 2026. However, please note that New York may be observing daylight saving time, which would make it 4 hours behind UTC, resulting in a time of 15:00:58 (3:00 PM).


In [25]:
response = agent_executor.invoke({"input": "What is todays date and current time in New York city?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_utc_date` with `{}`


Current date in UTC: 2026-11-03
Invoking: `get_current_utc_time` with `{}`


Current time in UTC: 19:01:21 UTCToday's date in UTC is November 3, 2026, and the current time in UTC is 19:01:21. 

To convert this to New York City time, which is in the Eastern Time Zone (ET), you need to subtract 5 hours for Eastern Standard Time (EST) or 4 hours for Eastern Daylight Time (EDT). Since it's November, EDT has ended, and EST is in effect. 

Thus, the current time in New York City is approximately 14:01 (2:01 PM) on November 3, 2026.

> Finished chain.
Today's date in UTC is November 3, 2026, and the current time in UTC is 19:01:21. 

To convert this to New York City time, which is in the Eastern Time Zone (ET), you need to subtract 5 hours for Eastern Standard Time (EST) or 4 hours for Eastern Daylight Time (EDT). Since it's November, EDT has ended, and EST is in effect. 

Thus, the current time in New York 

# Happy Learning